# Chapter 8 &mdash; Error-Correcting Design II: the NFA that Counts Dings

**Concept 7 of the Chapter 8 decomposition:** *Error-Correcting Design II: the NFA that Silently Corrects and Counts Dings*

When the wrong symbol arrives, correct it silently and move to a state layer recording one more ding.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Error-Correcting-NFA/Concept-Error-Correcting-NFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The same language, designed as a **machine** rather than an expression.

Read `0101` position by position. When the **expected** symbol arrives, advance within
the current layer. When the **wrong** symbol arrives, advance *and* drop to the next
layer &mdash; the layer index records how many "dings" (corrections) have occurred.

Layers $0, 1, 2$ are accepting at the end; there is no layer 3, so a third ding kills
the token.

This is **error-correcting decoding** in miniature, and the layered-state-name idea
recurs whenever you must count a bounded resource.

## 2. Definitions

### Generate the layered NFA

In [ ]:
TARGET = '0101'
MAXD = 2

def layered_nfa(target, maxd):
    lines = ['NFA']
    n = len(target)
    def nm(i, d):
        if i == 0 and d == 0: return 'I'
        return ('F' if i == n else 'S') + '_p%d_d%d' % (i, d)
    for i, want in enumerate(target):
        other = '1' if want == '0' else '0'
        for d in range(maxd + 1):
            lines.append('%s : %s -> %s' % (nm(i, d), want, nm(i+1, d)))
            if d < maxd:
                lines.append('%s : %s -> %s   !! ding %d' % (nm(i, d), other, nm(i+1, d+1), d+1))
    return md2mc('\n'.join(lines))

N = layered_nfa(TARGET, MAXD)
print("states :", len(N["Q"]), " final :", len(N["F"]))

### The reference

In [ ]:
def ham(a, b): return sum(x != y for x, y in zip(a, b))
def within(s, k=MAXD): return len(s) == len(TARGET) and ham(s, TARGET) <= k

<!-- nav-strip -->

---

&larr;&nbsp;[Ch8&nbsp;6.&nbsp;Error-Correcting Design I: the RE for "within Hamming Distance 2"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Hamming-Distance-RE/Concept-Hamming-Distance-RE.ipynb) &nbsp;&middot;&nbsp; [**Chapter 8** index](https://github.com/ganeshutah/Jove/blob/master/Chapter8-RE/README.md) &nbsp;&middot;&nbsp; [Ch8&nbsp;8.&nbsp;Cross-Checking Two Designs by Minimal-DFA Isomorphism](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Cross-Check-By-Isomorphism/Concept-Cross-Check-By-Isomorphism.ipynb)&nbsp;&rarr;

---

## 3. Tests

The layers are visible in the state names.

In [ ]:
for q in sorted(N["Q"]):
    print("  ", q)
print("\nthe _d<k> suffix IS the number of corrections made so far")

Accepts exactly the strings within distance 2.

In [ ]:
from itertools import product
acc = [''.join(p) for p in product('01', repeat=4) if accepts_nfa(N, ''.join(p))]
print("accepted (%d) :" % len(acc), acc)
assert set(acc) == {''.join(p) for p in product('01', repeat=4) if within(''.join(p))}
assert len(acc) == 11

A **third** ding has nowhere to go, so the token dies.

In [ ]:
worst = ''.join('1' if ch == '0' else '0' for ch in TARGET)   # distance 4
print("complement of the target :", worst, " distance", ham(worst, TARGET))
assert not accepts_nfa(N, worst)
three = '1011'          # 1-0, 0-1, 1-0 differ; the last symbol agrees
print("%r distance %d accepted? %s" % (three, ham(three, TARGET), accepts_nfa(N, three)))
assert ham(three, TARGET) == 3 and not accepts_nfa(N, three)
two = '1111'            # only two positions differ -- this one IS accepted
print("%r distance %d accepted? %s" % (two, ham(two, TARGET), accepts_nfa(N, two)))
assert ham(two, TARGET) == 2 and accepts_nfa(N, two)

Raising the layer count raises the tolerated distance, exactly.

In [ ]:
for k in range(0, 5):
    Nk = layered_nfa(TARGET, k)
    a = sum(1 for p in product('01', repeat=4) if accepts_nfa(Nk, ''.join(p)))
    from math import comb
    expect = sum(comb(4, j) for j in range(k+1))
    print("maxd=%d : %2d accepted, C(4,0..%d) sums to %2d" % (k, a, k, expect))
    assert a == expect

## 4. Animation

The layered machine: each row is one more correction.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. Make the dinging **non**-silent: emit a marker symbol. What machine class is that?
2. How many states for a 7-bit target at distance 3?
3. Why is the layer index bounded &mdash; and what would an unbounded one require?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter8-RE/Concept-Error-Correcting-NFA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')